# LangGraph MCP Tutorial - Chrome DevTools 브라우저 자동화

이 튜토리얼에서는 [`chrome-devtools-mcp`](https://github.com/ChromeDevTools/chrome-devtools-mcp) MCP 서버를 LangGraph와 Claude 모델을 사용하여 Chrome 브라우저를 자동화하고 디버깅하는 에이전트를 구축합니다.

## Chrome DevTools MCP 서버가 제공하는 도구

| 카테고리 | 도구 | 설명 |
|----------|------|------|
| **입력 자동화** | `click` | 페이지 요소 클릭 |
| | `drag` | 요소 드래그 |
| | `fill` | 입력 필드에 텍스트 입력 |
| | `fill_form` | 여러 폼 필드 동시 입력 |
| | `handle_dialog` | 브라우저 다이얼로그 처리 |
| | `hover` | 요소 위에 마우스 호버 |
| | `press_key` | 키보드 키 입력 |
| | `type_text` | 텍스트 타이핑 |
| | `upload_file` | 파일 업로드 |
| **페이지 탐색** | `navigate_page` | URL로 페이지 이동 |
| | `new_page` | 새 탭/페이지 생성 |
| | `list_pages` | 열린 모든 페이지 목록 조회 |
| | `select_page` | 특정 페이지 선택 |
| | `close_page` | 페이지 닫기 |
| | `wait_for` | 특정 조건이나 요소 대기 |
| **에뮬레이션** | `emulate` | 다양한 기기 에뮬레이션 |
| | `resize_page` | 뷰포트 크기 변경 |
| **성능 분석** | `start_trace` | 성능 트레이스 기록 시작 |
| | `stop_trace` | 트레이스 중지 및 분석 |
| | `analyze_insights` | 성능 인사이트 추출 |
| | `memory_snapshots` | 메모리 스냅샷 캡처 |
| **네트워크** | `get_network_request` | 특정 네트워크 요청 조회 |
| | `list_network_requests` | 모든 네트워크 요청 목록 조회 |
| **디버깅** | `evaluate_script` | 페이지 컨텍스트에서 JS 실행 |
| | `get_console_messages` | 콘솔 메시지 수집 |
| | `lighthouse_audit` | Lighthouse 감사 실행 |
| | `take_screenshot` | 페이지 스크린샷 캡처 |
| | `take_snapshot` | DOM 스냅샷 캡처 |

## 목차

1. [환경 설정](#환경-설정)
2. [Part 1: Chrome DevTools MCP 서버 연결](#part-1-chrome-devtools-mcp-서버-연결)
3. [Part 2: 기본 브라우저 탐색](#part-2-기본-브라우저-탐색)
4. [Part 3: 웹 상호작용 자동화](#part-3-웹-상호작용-자동화)
5. [Part 4: 디버깅 및 분석](#part-4-디버깅-및-분석)
6. [Part 5: 성능 분석](#part-5-성능-분석)
7. [Part 6: ToolNode를 활용한 커스텀 워크플로우](#part-6-toolnode를-활용한-커스텀-워크플로우)

## 사전 요구사항

- **Node.js v20.19+** 가 설치되어 있어야 합니다
- **Google Chrome** (stable 버전) 이 설치되어 있어야 합니다
- `ANTHROPIC_API_KEY`가 `.env` 파일에 설정되어 있어야 합니다
- 다음 명령으로 MCP 서버를 사전 설치할 수 있습니다: `npx -y chrome-devtools-mcp@latest`

## 환경 설정

환경 변수를 로드하고 LangSmith 로깅을 설정합니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)
# LangSmith 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph-V1-Tutorial


## 라이브러리 임포트 및 Windows 호환성 설정

MCP 클라이언트, LangGraph 에이전트, 그리고 Windows Jupyter 환경에서의 호환성 패치를 설정합니다.

In [2]:
import nest_asyncio
from typing import List, Dict, Any

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트: 다중 MCP 서버에 연결하여 도구를 가져옵니다
from langchain_mcp_adapters.client import MultiServerMCPClient

# 비동기 호환성을 활성화합니다 (Jupyter 환경에서 필요)
nest_asyncio.apply()

In [3]:
import sys, os, subprocess

# Windows + Jupyter workaround: MCP stdio passes Jupyter's sys.stderr to subprocess.Popen,
# but Jupyter's stderr doesn't support fileno(). Patch the default errlog to os.devnull.
if sys.platform == "win32":
    import mcp.client.stdio as _mcp_stdio

    _devnull_file = open(os.devnull, "w")

    # @asynccontextmanager wraps the original function — patch __wrapped__.__defaults__
    if hasattr(_mcp_stdio.stdio_client, "__wrapped__"):
        _mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_devnull_file,)

    # Also patch the helper that creates the subprocess
    _mcp_stdio._create_platform_compatible_process.__defaults__ = (
        None,
        _devnull_file,
        None,
    )


async def setup_mcp_client(server_configs: dict):
    """MCP 클라이언트를 설정하고 도구를 가져옵니다.

    Args:
        server_configs: 서버 구성 딕셔너리

    Returns:
        tuple: (MCP 클라이언트, 로드된 도구 목록)
    """
    client = MultiServerMCPClient(server_configs)
    tools = await client.get_tools()

    print(f"[MCP] {len(tools)}개의 도구가 로드되었습니다:")
    for tool in tools:
        print(f"  - {tool.name}")

    return client, tools

---

## Part 1: Chrome DevTools MCP 서버 연결

`chrome-devtools-mcp` 서버를 `npx`를 통해 stdio 전송 방식으로 연결합니다.

> **참고**: Node.js v20.19+ 와 Google Chrome이 설치되어 있어야 합니다.  
> 서버가 처음 실행될 때 `npx`가 패키지를 자동으로 다운로드합니다.

In [4]:
# Chrome DevTools MCP 서버 구성 (stdio 전송 방식, npx 사용)
server_configs = {
    "chrome-devtools": {
        "command": "npx",
        "args": ["-y", "chrome-devtools-mcp@latest", "--isolated"],
        "transport": "stdio",
    }
}

# MCP 클라이언트 초기화 및 도구 로드
client, tools = await setup_mcp_client(server_configs=server_configs)

[MCP] 29개의 도구가 로드되었습니다:
  - click
  - close_page
  - drag
  - emulate
  - evaluate_script
  - fill
  - fill_form
  - get_console_message
  - get_network_request
  - handle_dialog
  - hover
  - lighthouse_audit
  - list_console_messages
  - list_network_requests
  - list_pages
  - navigate_page
  - new_page
  - performance_analyze_insight
  - performance_start_trace
  - performance_stop_trace
  - press_key
  - resize_page
  - select_page
  - take_memory_snapshot
  - take_screenshot
  - take_snapshot
  - type_text
  - upload_file
  - wait_for


In [5]:
# Claude LLM 설정
llm = init_chat_model("claude-sonnet-4-6", temperature=0)

# 브라우저 자동화 에이전트의 시스템 프롬프트
BROWSER_SYSTEM_PROMPT = """
You are a browser automation agent using Chrome DevTools MCP tools.

Navigation guidelines:
- When calling navigate_page, always pass timeout=30000 to avoid premature timeouts.
- If a navigation timeout error occurs, do NOT stop — check list_pages to see if the page URL changed. If it did, the page loaded successfully despite the timeout.
- After navigating, always call list_pages to confirm the correct tab is selected. Use select_page if needed.
- Prefer waitUntil="domcontentloaded" over "load" for faster navigation on heavy pages.
- If a frame is detached, open a new_page and navigate there instead of retrying the old tab.
"""

# 에이전트 생성: Chrome DevTools MCP 도구를 사용하는 에이전트
agent = create_agent(
    llm,
    tools,
    system_prompt=BROWSER_SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),  # 대화 상태를 메모리에 저장
)

In [6]:
# 스트리밍 헬퍼 함수와 UUID 생성 함수를 import합니다
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

---

## Part 2: 기본 브라우저 탐색

Chrome 브라우저의 기본 탐색 기능을 사용하여 페이지 이동, 탭 관리, 기기 에뮬레이션을 수행합니다.

### 예제 1: 웹 페이지 탐색 및 스크린샷

`navigate_page` 도구로 URL에 접속하고 `take_screenshot`으로 현재 화면을 캡처합니다.

In [ ]:
# 대화 스레드 ID를 설정합니다
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 웹 페이지 탐색 및 스크린샷
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "https://www.python.org 페이지로 이동한 후 스크린샷을 찍어주세요. "
        "navigate_page 호출 시 timeout=30000 을 사용하세요. "
        "타임아웃 오류가 발생해도 list_pages 로 현재 페이지를 확인한 뒤 select_page 로 선택하고 스크린샷을 찍어주세요. "
        "페이지의 주요 내용도 간단히 설명해주세요."
    )]},
    config=config,
)

### 예제 2: 여러 탭 생성 및 관리

`new_page`, `list_pages`, `select_page`, `close_page` 도구를 사용하여 여러 탭을 동시에 관리합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 여러 탭 생성 및 관리
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "다음 작업을 순서대로 수행해주세요:\n"
        "1. 새 탭을 2개 생성하세요\n"
        "2. 첫 번째 새 탭에서 https://github.com 으로 이동하세요\n"
        "3. 두 번째 새 탭에서 https://docs.python.org 으로 이동하세요\n"
        "4. 현재 열려있는 모든 탭 목록을 보여주세요\n"
        "5. 각 탭의 URL과 제목을 정리해서 알려주세요"
    )]},
    config=config,
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
순서대로 작업을 수행하겠습니다!

## 1단계: 새 탭 2개 생성 및 URL 이동

### 예제 3: 모바일 기기 에뮬레이션

`emulate` 도구로 다양한 모바일 기기를 에뮬레이션하고, `resize_page`로 뷰포트를 조정한 후 `take_screenshot`으로 모바일 화면을 캡처합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 모바일 기기 에뮬레이션
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "다음 작업을 수행해주세요:\n"
        "1. iPhone 12 Pro 모바일 기기로 에뮬레이션하세요\n"
        "2. https://www.google.com 을 방문하세요\n"
        "3. 모바일 화면의 스크린샷을 찍어주세요\n"
        "4. 데스크톱 모드로 다시 전환하고 뷰포트를 1920x1080으로 조정한 후 스크린샷을 찍어서 비교해주세요"
    )]},
    config=config,
)

---

## Part 3: 웹 상호작용 자동화

폼 입력, 클릭, 키보드 조작 등 웹 페이지와의 상호작용을 자동화합니다.

### 예제 4: 검색 폼 자동 입력

`navigate_page`, `fill`, `click`, `wait_for`, `take_screenshot` 도구를 조합하여 검색 폼에 텍스트를 입력하고 검색 결과를 캡처합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 검색 폼 자동 입력
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "다음 작업을 수행해주세요:\n"
        "1. https://www.google.com 으로 이동하세요\n"
        "2. 검색창에 'LangGraph tutorial' 을 입력하세요\n"
        "3. Enter 키를 눌러 검색을 실행하세요\n"
        "4. 검색 결과가 로드될 때까지 기다린 후 스크린샷을 찍어주세요\n"
        "5. 상위 3개 검색 결과의 제목과 URL을 알려주세요"
    )]},
    config=config,
)

### 예제 5: 여러 폼 필드 자동 작성

`fill_form` 도구를 사용하면 여러 폼 필드를 한 번에 작성할 수 있습니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 여러 폼 필드 동시 작성
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "다음 작업을 수행해주세요:\n"
        "1. https://httpbin.org/forms/post 페이지로 이동하세요\n"
        "2. fill_form 도구를 사용하여 다음 필드들을 한 번에 입력하세요:\n"
        "   - custname: 'LangGraph 사용자'\n"
        "   - custtel: '010-1234-5678'\n"
        "   - custemail: 'langgraph@example.com'\n"
        "   - comments: 'Chrome DevTools MCP 튜토리얼 테스트'\n"
        "3. 입력 후 스크린샷을 찍어 폼이 잘 작성되었는지 확인해주세요"
    )]},
    config=config,
)

### 예제 6: 키보드 및 마우스 조작

`type_text`, `hover`, `press_key`, `drag` 도구를 사용하여 세밀한 키보드 및 마우스 조작을 수행합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 키보드 및 마우스 조작
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "다음 작업을 수행해주세요:\n"
        "1. https://www.w3schools.com/html/tryit.asp?filename=tryhtml_default 페이지로 이동하세요\n"
        "2. 페이지 상단의 'Run it' 버튼 위에 마우스를 호버해보세요\n"
        "3. 결과 프레임에서 JavaScript를 사용하여 'Hello, LangGraph!' 텍스트를 추가하세요\n"
        "4. Ctrl+A 단축키로 전체 선택, Ctrl+C 로 복사하는 키보드 조작을 수행해주세요\n"
        "5. 최종 화면의 스크린샷을 찍어주세요"
    )]},
    config=config,
)

### 예제 7: 파일 업로드 및 다이얼로그 처리

`upload_file` 도구로 파일을 업로드하고 `handle_dialog` 도구로 브라우저 다이얼로그를 처리합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 다이얼로그 처리
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "다음 작업을 수행해주세요:\n"
        "1. https://www.w3schools.com/jsref/tryit.asp?filename=tryjsref_alert 페이지로 이동하세요\n"
        "2. 페이지에서 JavaScript로 alert('Chrome DevTools MCP 테스트!') 를 실행하세요\n"
        "3. 나타나는 alert 다이얼로그를 확인(Accept)으로 처리해주세요\n"
        "4. 다이얼로그가 처리된 후 스크린샷을 찍어주세요"
    )]},
    config=config,
)

---

## Part 4: 디버깅 및 분석

JavaScript 실행, 콘솔 메시지 수집, 네트워크 분석 등 개발자 도구의 핵심 기능을 활용합니다.

### 예제 8: JavaScript 실행 및 DOM 검사

`evaluate_script` 도구로 페이지 컨텍스트에서 JavaScript를 실행하고, `take_snapshot`으로 DOM 구조를 캡처합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: JavaScript 실행 및 DOM 검사
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "https://www.python.org 페이지에서 다음 JavaScript 분석을 수행해주세요:\n"
        "1. 페이지 타이틀 (document.title) 가져오기\n"
        "2. 페이지에 있는 모든 링크(<a> 태그) 수 계산\n"
        "3. 페이지에 있는 모든 이미지(<img> 태그) 수 계산\n"
        "4. 현재 URL과 document.readyState 확인\n"
        "5. navigator.userAgent 값 가져오기\n"
        "6. DOM 스냅샷도 캡처해주세요\n"
        "결과를 정리하여 보고해주세요"
    )]},
    config=config,
)

### 예제 9: 콘솔 메시지 수집 및 분석

`evaluate_script`로 다양한 콘솔 메시지를 생성하고, `get_console_messages`로 소스 매핑된 스택 트레이스와 함께 수집합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 콘솔 메시지 생성 및 수집
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "빈 페이지(about:blank)에서 다음 작업을 수행해주세요:\n"
        "1. JavaScript를 실행하여 다음 콘솔 메시지들을 순서대로 출력하세요:\n"
        "   - console.log('정보: LangGraph MCP 테스트 시작')\n"
        "   - console.info('상세정보: Chrome DevTools MCP 연결 성공')\n"
        "   - console.warn('경고: 이것은 테스트 경고 메시지입니다')\n"
        "   - console.error('오류: 이것은 테스트 오류 메시지입니다')\n"
        "   - console.debug('디버그: 디버그 레벨 메시지')\n"
        "2. get_console_messages 도구를 사용하여 콘솔 메시지를 수집하세요\n"
        "3. 각 메시지의 레벨, 내용, 소스 위치를 분류하여 보고해주세요"
    )]},
    config=config,
)

### 예제 10: 네트워크 요청 분석

`list_network_requests`, `get_network_request` 도구를 사용하여 페이지 로드 시 발생하는 네트워크 트래픽을 캡처하고 분석합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 네트워크 요청 분석
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "https://httpbin.org/get 페이지를 방문하고 네트워크 분석을 수행해주세요:\n"
        "1. 페이지 로드 중 발생한 모든 네트워크 요청 목록을 가져오세요\n"
        "2. 각 요청의 URL, HTTP 메서드, 상태 코드를 정리하세요\n"
        "3. httpbin.org/get 에 대한 메인 API 요청의 상세 정보를 가져오세요\n"
        "4. 요청 헤더와 응답 내용을 분석하여 보고해주세요\n"
        "5. 요청별 응답 크기와 로드 시간도 포함하면 좋겠습니다"
    )]},
    config=config,
)

---

## Part 5: 성능 분석

Chrome DevTools의 강력한 성능 분석 도구를 활용하여 웹 사이트의 성능을 측정하고 개선점을 파악합니다.

### 예제 11: Lighthouse 성능 감사

`lighthouse_audit` 도구로 웹 페이지의 성능, 접근성, SEO, 모범 사례를 종합적으로 평가합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: Lighthouse 성능 감사
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "https://www.python.org 에 대한 Lighthouse 감사를 실행하고 종합 보고서를 작성해주세요:\n"
        "1. Lighthouse 감사를 실행하세요\n"
        "2. 다음 카테고리별 점수를 정리하세요:\n"
        "   - 성능(Performance) 점수와 주요 지표\n"
        "   - 접근성(Accessibility) 점수\n"
        "   - 모범 사례(Best Practices) 점수\n"
        "   - SEO 점수\n"
        "3. 개선이 필요한 주요 항목 3가지를 구체적인 해결 방안과 함께 제안해주세요"
    )]},
    config=config,
)

### 예제 12: 성능 트레이스 기록 및 분석

`start_trace`, `navigate_page`, `stop_trace`, `analyze_insights` 도구를 순서대로 사용하여 페이지 로드 중의 성능 트레이스를 기록하고 분석합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 성능 트레이스 기록 및 분석
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "https://github.com 페이지 로드 성능을 트레이스로 분석해주세요:\n"
        "1. 성능 트레이스 기록을 시작하세요 (start_trace)\n"
        "2. https://github.com 페이지로 이동하세요\n"
        "3. 페이지가 완전히 로드될 때까지 기다리세요\n"
        "4. 트레이스 기록을 중지하세요 (stop_trace)\n"
        "5. 성능 인사이트를 분석하세요 (analyze_insights)\n"
        "6. 다음 항목에 대한 분석 결과를 보고해주세요:\n"
        "   - 페이지 로드 시간 및 주요 성능 지표\n"
        "   - 가장 오래 걸린 작업 또는 리소스\n"
        "   - 성능 개선을 위한 구체적인 권장 사항"
    )]},
    config=config,
)

### 예제 13: 메모리 스냅샷 분석

`memory_snapshots` 도구를 사용하여 페이지의 메모리 사용 현황을 분석합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 메모리 스냅샷 분석
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "https://www.wikipedia.org 페이지의 메모리 사용 현황을 분석해주세요:\n"
        "1. 페이지로 이동하세요\n"
        "2. 페이지 로드 직후 메모리 스냅샷을 캡처하세요\n"
        "3. 검색창에 'Python programming language'를 입력하고 검색하세요\n"
        "4. 검색 결과 페이지에서 다시 메모리 스냅샷을 캡처하세요\n"
        "5. 두 스냅샷의 메모리 사용량을 비교하고:\n"
        "   - 총 메모리 사용량 변화\n"
        "   - 주요 메모리 소비 객체\n"
        "   - 잠재적인 메모리 누수 가능성\n"
        "   을 분석하여 보고해주세요"
    )]},
    config=config,
)

---

## Part 6: ToolNode를 활용한 커스텀 워크플로우

`ToolNode`를 사용하면 `create_agent` 대신 LangGraph `StateGraph`로 더 세밀한 제어가 가능한 워크플로우를 만들 수 있습니다.  
여기서는 웹사이트 종합 품질 검사 워크플로우를 구성합니다.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from typing import Annotated, TypedDict


class AgentState(TypedDict):
    """에이전트 상태 정의"""
    messages: Annotated[List[BaseMessage], add_messages]


async def create_devtools_workflow(server_configs: dict):
    """Chrome DevTools MCP 도구를 사용하는 커스텀 워크플로우를 생성합니다.

    Args:
        server_configs: MCP 서버 구성 딕셔너리

    Returns:
        CompiledStateGraph: 컴파일된 워크플로우 그래프
    """
    # MCP 클라이언트 생성 및 도구 로드
    client, tools = await setup_mcp_client(server_configs=server_configs)

    # Claude LLM 설정 및 도구 바인딩
    llm = init_chat_model("claude-sonnet-4-6", temperature=0)
    llm_with_tools = llm.bind_tools(tools)

    # 워크플로우 그래프 생성
    workflow = StateGraph(AgentState)

    async def agent_node(state: AgentState):
        """에이전트 노드: LLM을 호출하여 응답을 생성합니다"""
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    # ToolNode 생성: 도구 호출을 처리합니다
    tool_node = ToolNode(tools)

    # 그래프에 노드 추가
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", tool_node)

    # 엣지 정의: 시작 -> 에이전트
    workflow.add_edge(START, "agent")

    # 조건부 엣지: 에이전트 -> (도구 or 종료)
    workflow.add_conditional_edges("agent", tools_condition)

    # 도구 -> 에이전트 (도구 실행 후 다시 에이전트로)
    workflow.add_edge("tools", "agent")

    # 그래프 컴파일
    app = workflow.compile(checkpointer=InMemorySaver())

    return app


# 커스텀 워크플로우 생성
devtools_workflow = await create_devtools_workflow(server_configs)

### 워크플로우 그래프 시각화

In [ ]:
from IPython.display import Image, display

# 워크플로우 그래프를 시각화합니다
display(Image(devtools_workflow.get_graph().draw_mermaid_png()))

### 커스텀 워크플로우: 웹사이트 종합 품질 검사

생성된 커스텀 워크플로우를 사용하여 웹사이트의 성능, 접근성, 네트워크 효율성을 종합적으로 분석합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 커스텀 워크플로우 실행: 웹사이트 종합 품질 검사
response = await astream_graph(
    devtools_workflow,
    inputs={"messages": [("human",
        "https://www.python.org 웹사이트에 대한 종합 품질 검사를 수행하고 보고서를 작성해주세요:\n"
        "\n"
        "[1단계: 기본 탐색]\n"
        "- 페이지로 이동하여 스크린샷 캡처\n"
        "\n"
        "[2단계: 성능 분석]\n"
        "- Lighthouse 감사 실행하여 성능/접근성/SEO 점수 측정\n"
        "\n"
        "[3단계: 네트워크 분석]\n"
        "- 네트워크 요청 목록 분석\n"
        "- 요청 수, 총 데이터 전송량 확인\n"
        "\n"
        "[4단계: JavaScript 분석]\n"
        "- JS 실행으로 페이지 구조 (링크 수, 이미지 수, 스크립트 수) 파악\n"
        "\n"
        "[5단계: 최종 보고서]\n"
        "- 위 분석 결과를 종합하여 웹사이트 품질 평가 보고서 작성\n"
        "- 개선이 필요한 상위 3가지 항목과 구체적인 개선 방안 제시"
    )]},
    config=config,
)

---

## 정리

이 튜토리얼에서 다룬 내용:

### 핵심 기능 요약

| 카테고리 | 사용한 도구 | 주요 활용 사례 |
|----------|------------|----------------|
| **브라우저 탐색** | `navigate_page`, `new_page`, `list_pages`, `select_page`, `close_page` | 멀티탭 관리, 페이지 이동 |
| **에뮬레이션** | `emulate`, `resize_page` | 모바일/태블릿 화면 테스트 |
| **입력 자동화** | `fill`, `fill_form`, `click`, `type_text`, `hover`, `press_key` | 폼 작성, 검색, 키보드 조작 |
| **다이얼로그** | `handle_dialog`, `upload_file` | 팝업 처리, 파일 업로드 |
| **디버깅** | `evaluate_script`, `get_console_messages`, `take_screenshot`, `take_snapshot` | JS 실행, 로그 수집, 화면 캡처 |
| **네트워크** | `list_network_requests`, `get_network_request` | 트래픽 분석, API 디버깅 |
| **성능** | `lighthouse_audit`, `start_trace`, `stop_trace`, `analyze_insights`, `memory_snapshots` | 성능 측정, 메모리 분석 |

### 주요 포인트

- **chrome-devtools-mcp** 서버는 `npx -y chrome-devtools-mcp@latest` 명령으로 실행되며, Node.js와 Chrome이 필요합니다
- MCP 서버는 `MultiServerMCPClient`를 통해 LangGraph에 연결되며, 29개의 강력한 브라우저 자동화 도구를 제공합니다
- `create_agent`는 빠른 프로토타이핑에, `StateGraph` + `ToolNode`는 세밀한 제어가 필요한 복잡한 워크플로우에 적합합니다
- 같은 `thread_id`를 사용하면 대화 컨텍스트가 유지되어 여러 단계에 걸친 복잡한 브라우저 자동화 작업을 수행할 수 있습니다
- 성능 분석 도구(Lighthouse, 트레이스, 메모리 스냅샷)를 조합하면 웹 애플리케이션의 성능 병목을 효과적으로 찾아낼 수 있습니다